# QuEL-3 Experiment Configure Check

This notebook verifies the end-to-end `Experiment(...); exp.configure(...)` path for QuEL-3.

It shows that the `Experiment` configuration for `T1` contains readout targets `RQ00` through `RQ03`, then runs `exp.configure(...)`, and finally confirms that those targets were deployed through the QuEL-3 configuration manager.

Current temporary workaround: the notebook patches the QuEL-3 configuration manager so `exp.configure(...)` deploys only readout (`TRANSCEIVER`) requests. This avoids the current worker route-switch bug on control/pump ports while preserving the real `Experiment -> configure -> push -> deploy` flow for readout.


In [ ]:
from pathlib import Path

import qubex as qx
import qubex.system.system_manager as system_manager_module
from qubex.backend.quel3.infra.quelware_imports import (
    import_module_with_workspace_fallback,
)
from qubex.system.quel3 import Quel3SystemSynchronizer


In [ ]:
# Example selection
example_kind = "quel3"
if example_kind != "quel3":
    raise ValueError("This notebook is intended for the QuEL-3 example set.")

candidate_root_dirs = [
    Path.cwd(),
    Path.cwd() / "docs/examples/system",
    Path.cwd().parent,
]
example_root = next(
    (
        path
        for path in candidate_root_dirs
        if (path / "quel1" / "config").is_dir() and (path / "quel3" / "config").is_dir()
    ),
    None,
)
if example_root is None:
    raise FileNotFoundError(
        "Could not find `docs/examples/system/{quel1,quel3}` from the current working directory."
    )

example_dir = example_root / example_kind
config_dir = example_dir / "config"
params_dir = example_dir / "params"

# Hardware endpoint
server_host = "localhost"
server_port = 50051
use_standalone_client = True
standalone_unit_label = "quel3-02-a01"

# Configure only the T1 example box.
box_ids = ["T1"]

print(f"example_root: {example_root.resolve()}")
print(f"config_dir: {config_dir.resolve()}")
print(f"params_dir: {params_dir.resolve()}")
print(f"server: {server_host}:{server_port}")
print(f"use_standalone_client: {use_standalone_client}")
print(f"standalone_unit_label: {standalone_unit_label}")
print(f"box_ids: {box_ids}")


In [ ]:
exp = qx.Experiment(
    chip_id="64Q",
    qubits=["Q00", "Q01", "Q02", "Q03"],
    config_dir=config_dir,
    params_dir=params_dir,
)

print("backend_kind:", exp.system_manager.backend_kind)
print("loaded box_ids:", exp.ctx.box_ids)
print("experiment qubits:", exp.qubit_labels)


In [ ]:
t1_targets = {
    label: target
    for label, target in sorted(exp.experiment_system.gen_targets.items())
    if target.channel.port.box_id == "T1"
}
t1_readout_labels = tuple(
    label for label, target in t1_targets.items() if target.type.value == "READ"
)
t1_non_readout_labels = tuple(
    label for label, target in t1_targets.items() if target.type.value != "READ"
)

print("T1 generator targets:")
for label, target in t1_targets.items():
    print(
        f"- {label}: type={target.type.value}, port={target.channel.port.id}, frequency={target.frequency:.6f} GHz"
    )

print("T1 readout labels:", t1_readout_labels)
print("T1 readout target check:", t1_readout_labels == ("RQ00", "RQ01", "RQ02", "RQ03"))
print("T1 non-readout labels:", t1_non_readout_labels)
print(
    "Note: exp.configure(exclude=...) does not remove all control/pump targets, "
    "so this notebook patches deploy requests to readout-only instead."
)


In [ ]:
synchronizer = exp.system_manager._resolve_system_synchronizer()  # noqa: SLF001
if not isinstance(synchronizer, Quel3SystemSynchronizer):
    raise TypeError(f"Expected Quel3SystemSynchronizer, got {type(synchronizer)!r}")

manager = synchronizer.configuration_manager
original_build_requests = manager._build_instrument_deploy_requests  # noqa: SLF001

if use_standalone_client:
    client_module = import_module_with_workspace_fallback("quelware_client.client")
    create_standalone_client = client_module.create_standalone_client
    manager._load_quelware_client_factory = (  # noqa: SLF001
        lambda: (
            lambda endpoint, port: create_standalone_client(
                endpoint,
                port,
                unit_label=standalone_unit_label,
            )
        )
    )

manager._build_instrument_deploy_requests = (  # noqa: SLF001
    lambda *, experiment_system, box_ids: tuple(
        request
        for request in original_build_requests(
            experiment_system=experiment_system,
            box_ids=box_ids,
        )
        if request.role == "TRANSCEIVER"
    )
)

preview_requests = manager._build_instrument_deploy_requests(  # noqa: SLF001
    experiment_system=exp.experiment_system,
    box_ids=box_ids,
)
print("Patched exp.configure() to deploy readout-only requests:")
for request in preview_requests:
    print(
        f"- port_id={request.port_id}, role={request.role}, targets={request.target_labels}"
    )


In [ ]:
original_confirm_ask = system_manager_module.Confirm.ask
system_manager_module.Confirm.ask = lambda *args, **kwargs: True
try:
    exp.configure(
        box_ids=box_ids,
    )
finally:
    system_manager_module.Confirm.ask = original_confirm_ask

print("exp.configure() completed.")


In [ ]:
synchronizer = exp.system_manager._resolve_system_synchronizer()  # noqa: SLF001
if not isinstance(synchronizer, Quel3SystemSynchronizer):
    raise TypeError(f"Expected Quel3SystemSynchronizer, got {type(synchronizer)!r}")

manager = synchronizer.configuration_manager
target_alias_map = manager.target_alias_map
deployed_infos = manager.last_deployed_instrument_infos

rq_labels = ("RQ00", "RQ01", "RQ02", "RQ03")
rq_alias_map = {label: target_alias_map.get(label) for label in rq_labels}
rq_aliases = {alias for alias in rq_alias_map.values() if alias is not None}

print("RQ target -> alias map:")
for label in rq_labels:
    print(f"- {label}: {rq_alias_map[label]}")

print("unique aliases for RQ00-RQ03:", rq_aliases)
print("single transceiver alias check:", len(rq_aliases) == 1)

print("deployed instruments:")
for alias, infos in deployed_infos.items():
    print(f"- {alias}: instrument_count={len(infos)}")

if len(rq_aliases) == 1:
    alias = next(iter(rq_aliases))
    print("T1 readout deploy confirmed for:", rq_labels)
    print("deployed alias:", alias)
    print("deployed info count:", len(deployed_infos.get(alias, ())))
else:
    print("T1 readout deploy check failed.")
